# SceneVerse data check

Exploratory pass over the freshly-downloaded SceneVerse files
(`dataset/download_sceneverse.sh` / `.sbatch`) before writing any real
preprocessing. Two files so far:

- `source_data/sceneverse/scannet_scene_cap.json` — ScanNet scene-level captions
- `source_data/sceneverse/3rscan.zip` — unknown internal structure, not extracted yet

Goal: get real counts (not estimates) for "how many usable scenes does this
add" so we can decide if the scene-level-caption pretraining data pool is
big enough, and cross-reference against what's already confirmed
(`leo_annotations` 3RScan `scene_caption`, MMScan `Caption_region`).

**Adjust the path config in the first cell to match your actual cluster layout before running.**

In [ ]:
import json, zipfile
from pathlib import Path

# --- path config -- adjust these to your actual layout ---
ROOT = Path("/glob/g01-cache/pf/Yushuo/vjepa201")
SCENEVERSE_ROOT = ROOT / "source_data/sceneverse"
SCANNET_SCENE_CAP = SCENEVERSE_ROOT / "scannet_scene_cap.json"
THREERSCAN_ZIP = SCENEVERSE_ROOT / "3rscan.zip"

LEO_ANNO = ROOT / "source_data/leo_annotations/annotations"
SCANNET_POSED = ROOT / "source_data/scannet/posed_images"
THREERSCAN_ROOT = ROOT / "source_data/3rscan"

for name, p in [
    ("scannet_scene_cap", SCANNET_SCENE_CAP),
    ("3rscan.zip", THREERSCAN_ZIP),
    ("leo_anno", LEO_ANNO),
    ("scannet_posed", SCANNET_POSED),
    ("3rscan_raw", THREERSCAN_ROOT),
]:
    print(("OK  " if p.exists() else "MISSING  ") + f"{name:<18} {p}")

## 1. `3rscan.zip` -- list contents (not extracted)

We don't know what's inside yet: could be a straight repackage of leo's
`3rscan_scenecap_{train,val}.json`, could be something new (per-region,
different captions, extra metadata). List every entry + total size first,
extract nothing to disk.

In [ ]:
with zipfile.ZipFile(THREERSCAN_ZIP) as z:
    infos = z.infolist()

total_bytes = sum(i.file_size for i in infos)
print(f"{len(infos)} entries, {total_bytes / 1e6:.1f} MB uncompressed total\n")

# group by extension to see what kinds of files are in here
from collections import Counter
ext_counts = Counter(Path(i.filename).suffix for i in infos)
print("by extension:", dict(ext_counts))

print("\nfirst 30 entries:")
for i in infos[:30]:
    print(f"  {i.file_size:>12} bytes  {i.filename}")

## 2. Peek inside the zip's json/text files without extracting

For every `.json` (or other small text-like) entry, read it straight from
the zip (`ZipFile.open`, in-memory) and print its top-level structure --
dict keys / list length / one example item. Skips anything over ~20MB
(read as raw bytes for a byte-count instead) so we don't accidentally load
something huge into memory.

In [ ]:
SIZE_CAP = 20_000_000  # 20MB

def describe(obj, max_items=6):
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"  dict, {len(keys)} keys, first keys: {keys[:max_items]}")
        if keys:
            k0 = keys[0]
            print(f"  obj[{k0!r}] = {str(obj[k0])[:400]}")
    elif isinstance(obj, list):
        print(f"  list, {len(obj)} items")
        if obj:
            print(f"  obj[0] = {str(obj[0])[:400]}")
    else:
        print(f"  {type(obj)}: {str(obj)[:400]}")

with zipfile.ZipFile(THREERSCAN_ZIP) as z:
    text_like = [i for i in z.infolist() if i.filename.lower().endswith((".json", ".txt", ".csv"))]
    print(f"{len(text_like)} json/txt/csv entries found\n")
    for i in text_like:
        print(f"=== {i.filename}  ({i.file_size} bytes) ===")
        if i.file_size > SIZE_CAP:
            print(f"  skipped (over {SIZE_CAP/1e6:.0f}MB cap)")
            continue
        raw = z.read(i.filename)
        if i.filename.lower().endswith(".json"):
            try:
                obj = json.loads(raw)
                describe(obj)
            except json.JSONDecodeError as e:
                print(f"  not valid single-document JSON ({e}); first 300 bytes:")
                print(" ", raw[:300])
        else:
            print(" ", raw[:300])
        print()

## 3. `scannet_scene_cap.json` -- scene count, captions/scene, one example

In [ ]:
scannet_cap = json.load(open(SCANNET_SCENE_CAP))

n_scenes = len(scannet_cap)
n_captions = sum(len(v["captions"]) for v in scannet_cap.values())
print(f"scenes: {n_scenes}")
print(f"total captions: {n_captions}")
print(f"avg captions/scene: {n_captions / n_scenes:.2f}")

example_id = next(iter(scannet_cap))
print(f"\nexample scene_id: {example_id}")
print("example caption:", scannet_cap[example_id]["captions"][0][:400])

## 4. Cross-reference `scannet_scene_cap` scene_ids against local posed_images

How many of these scene_ids actually have frames on disk (`source_data/scannet/posed_images/<scene_id>/`)?

In [ ]:
cap_ids = set(scannet_cap.keys())
posed_ids = {p.name for p in SCANNET_POSED.iterdir() if p.is_dir()} if SCANNET_POSED.exists() else set()

overlap = cap_ids & posed_ids
print(f"scannet_scene_cap scene_ids: {len(cap_ids)}")
print(f"posed_images scene dirs on disk: {len(posed_ids)}")
print(f"overlap (usable now): {len(overlap)}")
print(f"caption-only, no frames found: {len(cap_ids - posed_ids)}")
if cap_ids - posed_ids:
    print("  example missing:", sorted(cap_ids - posed_ids)[:5])

## 5. Compare against `leo_annotations`'s 3RScan `scene_caption` (already confirmed earlier)

Reload leo's train+val 3RScan scene captions here too, so this notebook alone
gives the full tally without cross-referencing an older chat message.

In [ ]:
leo_3rscan_files = {
    "train": LEO_ANNO / "alignment/scene_caption/3rscan_scenecap_train.json",
    "val": LEO_ANNO / "alignment/scene_caption/3rscan_scenecap_val.json",
}

leo_3rscan = {}
for split, path in leo_3rscan_files.items():
    if not path.exists():
        print(f"[skip] {split}: not found at {path}")
        continue
    data = json.load(open(path))
    leo_3rscan.update(data)
    print(f"{split}: {len(data)} scenes, {sum(len(v) for v in data.values())} caption entries")

print(f"\nleo 3RScan total unique scenes (train+val union): {len(leo_3rscan)}")

# cross-check against raw 3RScan scan dirs on disk
rscan_dirs = {p.name for p in THREERSCAN_ROOT.iterdir() if p.is_dir()} if THREERSCAN_ROOT.exists() else set()
print(f"3RScan scan dirs on disk: {len(rscan_dirs)}")
print(f"leo caption <-> raw scan overlap: {len(set(leo_3rscan) & rscan_dirs)}")

## 6. Final tally

Pulls together every scene-level-caption source checked so far (this
notebook + the MMScan `Caption_region` numbers already confirmed via the
remote report) into one summary. Update the `MMSCAN_*` constants below if
those numbers have changed since.

In [ ]:
# From the earlier MMScan_Caption_region.json inspection (region-level, not
# scene-level -- one region entry stands on its own as a training sample,
# see the analysis in chat for why "single region" filtering isn't required).
# Update these if you re-run that inspection and get different numbers.
MMSCAN_REGION_TRAIN = 4667
MMSCAN_REGION_VAL = 1191

print("=== Scene/region-level caption pool ===")
print(f"leo 3RScan (scene-level, train+val union):        {len(leo_3rscan):>6} scenes")
print(f"sceneverse ScanNet (scene-level):                  {n_scenes:>6} scenes  ({len(overlap)} with frames on disk)")
print(f"MMScan Caption_region (region-level, all 3 datasets, train+val): {MMSCAN_REGION_TRAIN + MMSCAN_REGION_VAL:>6} region entries")
print()
print("Re-run section 1/2 once the 3rscan.zip structure is known, and update")
print("this cell with whatever new scene count it adds (may overlap with leo's")
print("3RScan scenes above -- check scan_id overlap before treating it as pure addition).")
